In [ ]:
import os, json
import pickle
import pandas as pd 
import numpy as np
from termcolor import colored
pd.set_option('display.max_colwidth', None)


In [ ]:
import os
os.chdir(".")
print(os.getcwd())

In [ ]:
import google.generativeai as genai
from google.generativeai import GenerationConfig

In [ ]:
import matplotlib.pyplot as plt


In [ ]:
gold_hedge_df = pd.read_csv("./exp3_rewriting/hedge_extraction/data/survey-results.csv", index_col=None)   # this CSV must be obtained from: https://github.com/wadefagen/datasets/blob/cc85869f8df3453a3f2f9bd3c7d3272037936b61/Perception-of-Probability-Words/survey-results.csv
gold_hedge_df.drop(gold_hedge_df.columns[-3:], axis=1, inplace=True)
gold_hedge_df.head(2)

In [ ]:
hedges = [
    x.replace('"',"").lower().replace("we", "I") if "Believe" in x else x.replace('"',"").lower()
    for x in gold_hedge_df.columns.tolist()
]
hedges

In [ ]:
annotations_df = pd.read_csv("./exp3_rewriting/hedge_extraction/data/sampled_10k_sentences_annotations.csv") # this CSV must be obtained from https://arxiv.org/pdf/2509.24202
annotations_df['mean_score'] = annotations_df[['annotater_confidence_score1', 'annotater_confidence_score2', 'annotater_confidence_score3', 'annotater_confidence_score4', 'annotater_confidence_score5']].mean(axis=1)
annotations_df['ratings'] = annotations_df[['annotater_confidence_score1', 'annotater_confidence_score2', 'annotater_confidence_score3', 'annotater_confidence_score4', 'annotater_confidence_score5']].values.tolist()
samples_df = annotations_df[['ground_truth_answer', 'uncertainty_expression', 'mean_score', 'ratings', 'llm_confidence_level', 'source_llm']]
assert(len(samples_df)==len(annotations_df))
samples_df.head(20)


In [ ]:
EXTRACTION_PROMPT = """You are a linguistic expert. You will be given an answer and a text which expresses that answer using one or more sentences, each of which contains zero or more linguistic uncertainty expressions (also known as hedge words, or hedge phrases). Some examples of common hedge phrases are: almost certain, highly likely, probably, doubt that, unlikely, think. Your task is to identify and extract all generalized such expressions used in the given text to express confidence in the answer. Make sure the hedge expressions extracted are generalized: DO NOT include the Answer in the extracted hedge phrase or any information specific to the sentence which uses the hedge. Your extracted hedges should be able to be used in a plug-and-play fashion in any new sentence to express uncertainty linguistically. Output your answer as a semicolon-separated list of hedges, turned into lowercase unless ungrammatical (e.g., 'I' should be capitalized), with no other text. If no hedge is used respond nothing. End your entire response with ####

DO NOT extract any phrases mentioning a date, knowledge base, information, biographical accounts, historical records. DO NOT extract “yes” or “no” as a hedge phrase. 
BAD examples representing the types of hedge phrases you should AVOID extracting include: At the time of that publication (too specific, mentions answer), he is known for (too specific), historical documentation confirms (too specific), Arnold is a possibility (mentions answer), not available in my current knowledge base, most biographical accounts indicate (too specific, suggests answer is a person)
These are all NOT generalizable, i.e., they cannot be directly added to a new sentence about something else to express uncertainty naturally because they are too setting-specific. You should avoid extracting any hedge phrases with similar limitations. If you do not adhere to this, you will lose your job.

Answer: 13000
Text: I think the Warcraft wiki says 13,000 years, but I could be mistaken.
Hedges: I think; I could be mistaken ####

Answer: Islamia College of Science and Commerce
Text: Based on available information, Islamia College of Science and Commerce was probably the college accredited by the UGC in April 2010. I am not too sure.
Hedges: based on the available information; probably; I am not too sure ####

Answer: Linus Pauling
Text: All signs point to Linus Pauling.
Hedges: all signs point to ####

Answer: 5109m
Text: The Mediterranean Sea's maximum depth measures 5,109 meters.
Hedges: ####

Answer: May 10, 2023
Text: To the best of my recollection, it was May 10, 2023, when they made the announcement.
Hedges: to the best of my recollection ####

Answer: 13 May 2004
Text: I believe it was 13 May 2004 when he received the appointment.
Hedges: I believe ####

Answer: Ken Palmer
Text: That information isn’t available to me, so I can’t respond with certainty.
Hedges: I can't respond with certainty ####

Answer: 80
Text: Perhaps he was 80 years old.	
Hedges: perhaps ####

Answer: May 10, 1996
Text: There’s a chance that the private launch happened on May 10, 1996.
Hedges: there's a chance ####

Answer: 25 April 2011
Text: I’m almost certain I’m wrong, but maybe *Human Planet* first aired on 25 April 2011 on Discovery en Español.
Hedges: I'm almost certain I'm wrong; maybe ####

Answer: today
Text: I’m fairly sure his birthday is today
Hedges: I'm fairly sure ####

Answer: 4
Text: It's clear that the American Classical Music Hall of Fame inducted four people this year. I know this for a fact.
Hedges: it's clear that; I know for a fact ####

Answer: yes
Text: My tentative answer is yes
Hedges: my tentative answer is ####

Answer: 50
Text: She might have been 50 when she passed away.
Hedges: might ####

Answer: no
Text: As of 2022, the answer was no.
Hedges: ####

Answer: Emmett
Text: Emmett is a possibility
Hedges: is a possibility

Answer: 1964
Text: 1964 is what comes to mind
Hedges: is what comes to mind ####

Answer: August 2011
Text: August 2011 sounds familiar
Hedges: sounds familiar ####

Answer: 10
Text: Historical documentation confirms the answer is 10.
Hedges: ####

Answer: {answer}
Text: {text}
Hedges: <your comma-separated list here>"""

In [ ]:
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))
model = genai.GenerativeModel("gemini-2.5-flash-lite")

def get_hedges_from_text(answer, text):
    
    generation_config = GenerationConfig(
        max_output_tokens=100, 
        candidate_count=1,
        stop_sequences=['Answer:'],
        temperature=0,
    )
            
    # Format Prompt & Get Response
    prompt = EXTRACTION_PROMPT.format(
        answer=answer,
        text=text,
    )

    try:
        response = model.generate_content(
            prompt, 
            generation_config=generation_config,
        ).text
    except Exception as e: 
        print(colored("Error extracting hedges for sentence:", "red"), text, "\n", e)
        response = ""

    # Process Response
    try:
        response = response.split("####")[0]
    except Exception as e:
        print(colored("Error parsing response:", "red"), response, "\n", e)
        response = "" 
    try:
        hedges = [x.strip() for x in response.split(";")]
    except Exception as e:
        print(colored("Error splitting hedges:", "red"), response, "\n", e)
        hedges = []

    return hedges

def get_hedges_from_texts(answers, texts, slice, dev_mode=False, restart=False):

    checkpoint_dir = "./exp3_rewriting/hedge_extraction/checkpoints"
    os.makedirs(os.path.join(checkpoint_dir), exist_ok=True)
    checkpoint_path = os.path.join(checkpoint_dir, f"slice_{slice}.pkl")

    if dev_mode:
        answers = answers[:100]
        texts = texts[:100]
    else: 
        if slice<9:
            answers = answers[1000*slice:1000*(slice+1)]
            texts = texts[1000*slice:1000*(slice+1)]
        else: 
            answers = answers[1000*slice:]
            texts = texts[1000*slice:]

    if os.path.exists(checkpoint_path) and not restart:

        with open(checkpoint_path, "rb") as f:
            ckpt = pickle.load(f)

        start_idx = ckpt["idx"] + 1
        extracted_hedges = ckpt["extracted_hedges"]
        print(f"Resuming from checkpoint at index {start_idx}...")
        
    else:
        start_idx = 0
        extracted_hedges = []
        print(f"No checkpoint found for slice {slice}, starting fresh...")

    for idx, (a, t) in enumerate(zip(answers[start_idx:], texts[start_idx:]), start=start_idx):

        print(f"Getting hedges for sample index {idx} of {len(answers)} (slice {slice})")
        hedges_for_sample = get_hedges_from_text(a, t)
        extracted_hedges.append(hedges_for_sample)

        with open(checkpoint_path, "wb") as f:
            pickle.dump({"idx": idx, "extracted_hedges": extracted_hedges}, f)

    return extracted_hedges


In [ ]:
answers = samples_df.ground_truth_answer.tolist()
texts = samples_df.uncertainty_expression.tolist()

In [ ]:
extracted_hedges0 = get_hedges_from_texts(answers, texts, slice=0, restart=False)
extracted_hedges1 = get_hedges_from_texts(answers, texts, slice=1, restart=False)
extracted_hedges2 = get_hedges_from_texts(answers, texts, slice=2, restart=False)
extracted_hedges3 = get_hedges_from_texts(answers, texts, slice=3, restart=False)
extracted_hedges4 = get_hedges_from_texts(answers, texts, slice=4, restart=False)
extracted_hedges5 = get_hedges_from_texts(answers, texts, slice=5, restart=False)
extracted_hedges6 = get_hedges_from_texts(answers, texts, slice=6, restart=False)
extracted_hedges7 = get_hedges_from_texts(answers, texts, slice=7, restart=False)
extracted_hedges8 = get_hedges_from_texts(answers, texts, slice=8, restart=False)
extracted_hedges9 = get_hedges_from_texts(answers, texts, slice=9, restart=False)
# extracted_hedges = get_hedges_from_texts(answers, texts, slice=0, dev_mode=True, restart=False)

In [ ]:
extracted_hedges = extracted_hedges0 + extracted_hedges1 + extracted_hedges2 + extracted_hedges3 + extracted_hedges4 + extracted_hedges5 + extracted_hedges6 + extracted_hedges7 + extracted_hedges8 + extracted_hedges9
len(extracted_hedges)

In [ ]:
samples_df_test = samples_df.head(len(extracted_hedges)) #.head(15)
samples_df_test['extracted_hedges'] = extracted_hedges
exploded_df = samples_df_test.explode('extracted_hedges')
exploded_df["extracted_hedges"] = exploded_df["extracted_hedges"].str.replace("’", "'")

agg_dict = {
    'mean_score': 'mean', 
    'ratings': lambda x: [item for sublist in x for item in sublist],
    # 'n': 'size',
}
for col in samples_df_test.columns:
    if col not in ['extracted_hedges', 'mean_score', 'ratings']:
        agg_dict[col] = list

# Create result df with one hedge per row
result_df = exploded_df.groupby('extracted_hedges', as_index=False).agg(agg_dict)
result_df['n'] = exploded_df.groupby('extracted_hedges').size().values
result_df = result_df.rename(columns={'extracted_hedges': 'hedge'})

# Add statistical columns based on the ratings column
result_df['min'] = result_df['ratings'].apply(lambda x: np.min(x) if len(x) > 0 else np.nan)
result_df['max'] = result_df['ratings'].apply(lambda x: np.max(x) if len(x) > 0 else np.nan)
result_df['med'] = result_df['ratings'].apply(lambda x: np.median(x) if len(x) > 0 else np.nan)
result_df['q1'] = result_df['ratings'].apply(lambda x: np.percentile(x, 25) if len(x) > 0 else np.nan)
result_df['q3'] = result_df['ratings'].apply(lambda x: np.percentile(x, 75) if len(x) > 0 else np.nan)
result_df['iqr'] = result_df['q3'] - result_df['q1']
result_df = result_df[['hedge', 'mean_score', 'ground_truth_answer', 'uncertainty_expression', 'llm_confidence_level', 'source_llm', 'ratings', 'n', 'min', 'max', 'med', 'q1', 'q3', 'iqr']]

In [ ]:
result_df = result_df[result_df["hedge"] != "<your comma-separated list here>"]
result_df = result_df[~result_df["hedge"].str.contains("information")]
result_df = result_df[~result_df["hedge"].str.contains("He")]
result_df = result_df[~result_df["hedge"].str.contains("She")]
result_df = result_df[~result_df["hedge"].str.contains("answer")]
result_df = result_df[~result_df["hedge"].str.contains("uncertainty")]
result_df = result_df[~result_df["hedge"].str.contains("historical")]
result_df = result_df[~result_df["hedge"].str.contains("data")]
result_df = result_df[~result_df["hedge"].str.contains("records")]
result_df = result_df[~result_df["hedge"].str.contains("base")]
result_df = result_df[~result_df["hedge"].str.contains("figure")]
result_df = result_df[~result_df["hedge"].str.contains("%")]
result_df = result_df[~result_df["hedge"].str.contains("confidence")]
result_df = result_df[~result_df["hedge"].str.contains("biograph")]
result_df = result_df[~result_df["hedge"].str.contains("account")]
result_df = result_df[~result_df["hedge"].str.contains("histor")]
result_df = result_df[~result_df["hedge"].str.contains("pass")]
result_df = result_df[~result_df["hedge"].str.contains("vicinity")]
result_df = result_df[~result_df["hedge"].str.contains("name")]
result_df = result_df[~result_df["hedge"].str.contains("person")]
result_df = result_df[~result_df["hedge"].str.contains("they")]
result_df = result_df[~result_df["hedge"].str.contains("rumor")]
result_df = result_df[~result_df["hedge"].str.contains("available")]
result_df = result_df[~result_df["hedge"].str.contains("knowledge")]
result_df = result_df[~result_df["hedge"].str.contains("source")]
result_df = result_df[~result_df["hedge"].str.contains("you")]
result_df = result_df[~result_df["hedge"].str.contains("sorry")]
result_df = result_df[~result_df["hedge"].str.contains("refrain")]
result_df = result_df[~result_df["hedge"].str.contains("report")]
result_df = result_df[~result_df["hedge"].str.contains("theory")]
result_df = result_df[~result_df["hedge"].str.contains("double-check")]
result_df = result_df[~result_df["hedge"].str.contains("check")]
result_df = result_df[~result_df["hedge"].str.contains("known as")]
result_df = result_df[~result_df["hedge"].str.contains("indicates")]
result_df = result_df[~result_df["hedge"].str.contains("approximat")]
result_df = result_df[~result_df["hedge"].str.contains("ballpark")]
result_df = result_df[~result_df["hedge"].str.contains("reference")]
result_df = result_df[~result_df["hedge"].str.contains("year")]
result_df = result_df[~result_df["hedge"].str.contains("categorically")]
result_df = result_df[~result_df["hedge"].str.contains("derive")]
result_df = result_df[~result_df["hedge"].str.contains("have")]
result_df = result_df[~result_df["hedge"].str.contains("been")]
result_df = result_df[~result_df["hedge"].str.contains("Arnold")]
result_df = result_df[~result_df["hedge"].str.contains("publication")]
result_df = result_df[~result_df["hedge"].str.contains("paper")]
result_df = result_df[~result_df["hedge"].str.contains("research")]
result_df = result_df[~result_df["hedge"].str.contains("Emmett")]
result_df = result_df[~result_df["hedge"].str.contains("August")]
result_df = result_df[~result_df["hedge"].str.contains("2")]
result_df = result_df[~result_df["hedge"].str.contains("0")]
result_df = result_df[~result_df["hedge"].str.contains("1")]
result_df = result_df[~result_df["hedge"].str.contains("3")]
result_df = result_df[~result_df["hedge"].str.contains("4")]
result_df = result_df[~result_df["hedge"].str.contains("5")]
result_df = result_df[~result_df["hedge"].str.contains("6")]
result_df = result_df[~result_df["hedge"].str.contains("7")]
result_df = result_df[~result_df["hedge"].str.contains("8")]
result_df = result_df[~result_df["hedge"].str.contains("9")]
result_df = result_df[~result_df["hedge"].str.contains("istorical")]
result_df = result_df[~result_df["hedge"].str.contains("documentation")]
result_df = result_df[~result_df["hedge"].str.contains("yes")]
result_df = result_df[~result_df["hedge"].str.contains("retrieve")]
result_df = result_df[~result_df["hedge"].str.contains("number")]
result_df = result_df[~result_df["hedge"].str.contains("apolo")]
result_df = result_df[~result_df["hedge"].str.contains("document")]
result_df = result_df[~result_df["hedge"].str.contains("pinpoint")]
result_df = result_df[~result_df["hedge"].str.contains("unable to provide")]
result_df = result_df[~result_df["hedge"].str.contains("must admit")]
result_df = result_df[~result_df["hedge"].str.contains("won't hazard")]
result_df = result_df[~result_df["hedge"].str.contains("help")]
result_df = result_df[~result_df["hedge"].str.contains("date")]
result_df = result_df[~result_df["hedge"].str.contains("Sea")]
result_df = result_df[~result_df["hedge"].str.contains("question")]
result_df = result_df[~result_df["hedge"].str.contains("orry")]
result_df = result_df[~result_df["hedge"].str.contains("indication")]
result_df = result_df[~result_df["hedge"].str.contains("about")]
result_df = result_df[~result_df["hedge"].str.contains("around")]
result_df = result_df[~result_df["hedge"].str.contains("appears")]
result_df = result_df[~result_df["hedge"].str.contains("current capabilities")]
result_df = result_df[~result_df["hedge"].str.contains("clearly ")]
result_df = result_df[~result_df["hedge"].str.contains("close to")]
result_df = result_df[~result_df["hedge"].str.contains("commonly attributed")]
result_df = result_df[~result_df["hedge"].str.contains("could be roughly")]
result_df = result_df[~result_df["hedge"].str.contains("could be somewhere around")]
result_df = result_df[~result_df["hedge"].str.contains("could be around")]
result_df = result_df[~result_df["hedge"].str.contains("could fit")]
result_df = result_df[~result_df["hedge"].str.contains("could fit")]
result_df = result_df[~result_df["hedge"].str.contains("could it")]
result_df = result_df[~result_df["hedge"].str.contains("could potentially be attributed")]
result_df = result_df[~result_df["hedge"].str.contains("eludes")]
result_df = result_df[~result_df["hedge"].str.contains("distinction")]
result_df = result_df[~result_df["hedge"].str.contains("honestly")]
result_df = result_df[~result_df["hedge"].str.contains("wiki")]
result_df = result_df[~result_df["hedge"].str.contains("is indeed")]
result_df = result_df[~result_df["hedge"].str.contains("sticking")]
result_df = result_df[~result_df["hedge"].str.contains("speculation")]
result_df = result_df[~result_df["hedge"].str.contains("brain")]
result_df = result_df[~result_df["hedge"].str.contains("memory is")]
result_df = result_df[~result_df["hedge"].str.contains("notes suggest")]
result_df = result_df[~result_df["hedge"].str.contains("memory's")]
result_df = result_df[~result_df["hedge"].str.contains("finding")]
result_df = result_df[~result_df["hedge"].str.contains("or so")]
result_df = result_df[~result_df["hedge"].str.contains("identified")]
result_df = result_df[~result_df["hedge"].str.contains("maximum")]
result_df = result_df[~result_df["hedge"].str.contains("depth")]
result_df = result_df[~result_df["hedge"].str.contains("there seems to be")]
result_df = result_df[~result_df["hedge"].str.contains("it's a weak")]
# result_df = result_df[~result_df["hedge"].str.contains("")]
# result_df = result_df[~result_df["hedge"].str.contains("")]
# result_df = result_df[~result_df["hedge"].str.contains("")]
# result_df.tail(20)

In [ ]:
from pprint import pprint 
pprint(result_df.hedge.unique().tolist())

In [ ]:
result_df.hedge.unique().shape

In [ ]:
result_df.n.unique()

In [ ]:
hedge_stats = result_df.copy()

topN = 101
top = hedge_stats.sort_values("n", ascending=False).head(topN-1)

plt.figure(figsize=(10, 15))
plt.barh(top["hedge"], top["n"])
plt.xlabel("Count (events)")
plt.title(f"Top {topN-1} hedge phrases by frequency")
max_n = int(top["n"].max())
plt.xticks(np.arange(0, max_n + 1, 50))
# plt.xticks(np.arange(0, 50, 5))
plt.tight_layout()
plt.show()

In [ ]:
top_phrases = hedge_stats.head(topN)["hedge"].tolist()

order = (hedge_stats[hedge_stats["hedge"].isin(top_phrases)]
         .groupby("hedge")["mean_score"].median()
         .sort_values(ascending=True).index.tolist())

data = [hedge_stats[hedge_stats["hedge"] == h]["ratings"].iloc[0] for h in order]

plt.figure(figsize=(11, 13))
plt.boxplot(data, vert=False, labels=order, showmeans=True, meanprops={'marker': "*"}) #meanline=True, #showfliers=True,
plt.xlabel("Human perceived probability (mean across annotators)")
plt.title("Rating distributions for top hedge phrases (boxplots, no outliers)")
plt.xticks(np.arange(0, 101, 10))
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(hedge_stats["n"], hedge_stats["mean_score"], alpha=0.4)
plt.xlabel("Hedge count per response (merged)")
plt.ylabel("Human perceived probability (mean)")
plt.title("Hedging density vs perceived probability")
max_n = int(hedge_stats["n"].max())
plt.xticks(np.arange(0, max_n + 1, 50))
plt.tight_layout()
plt.show()


In [ ]:
top = hedge_stats.head(topN).copy()
top = top.sort_values("mean_score", ascending=True)

sizes = 50 + 450 * (top["n"] / top["n"].max())

plt.figure(figsize=(10, 12))
plt.scatter(top["mean_score"], range(len(top)), s=sizes, alpha=0.7)
plt.yticks(range(len(top)), top["hedge"])
plt.xlabel("Mean human rating")
plt.title("Top hedge expressions: mean rating (bubble size = frequency)")
plt.tight_layout()
plt.show()
